In [ ]:
from google.colab import files
files.upload()

Saving adversarial_training.py to adversarial_training.py


{'adversarial_training.py': b'"""\nAdversarial training with PGD-5 augmentation.\n\nDefense against evasion attacks (FGSM, PGD).\nRuns on Google Colab (T4 GPU). Saves baseline_robust.pt and\nrobustness_curve.json to /content/.\n\nMITRE ATLAS: AML.T0043 (Craft Adversarial Data)\nNIST AI RMF: MANAGE (MG-2), MEASURE (MS-2)\n"""\nimport os\nimport sys\nimport time\nimport json\nimport random\nfrom datetime import datetime\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader, Subset, random_split\nfrom torchvision import datasets, transforms\n\nsys.path.insert(0, "/content")\nfrom net import SmallCNN, CLASSES, CIFAR_MEAN, CIFAR_STD\n\n\n# -----------------------------------------------------------\n# Configuration\n# -----------------------------------------------------------\nTARGET_LABELS = [0, 1, 2, 3]\nOUT_PATH = "/content/baseline_robust.pt"\nCURVE_PATH = "/content/robustness_cu

In [ ]:
!ls -la /content/adversarial_training.py
!ls -la /content/net.py
!ls /content/data/cifar-10-batches-py/ | head -3

-rw-r--r-- 1 root root 10605 Sep 24 06:20 /content/adversarial_training.py
ls: cannot access '/content/net.py': No such file or directory
ls: cannot access '/content/data/cifar-10-batches-py/': No such file or directory


In [ ]:
%%writefile /content/net.py
"""
SmallCNN for 4-class CIFAR-10 subset.

Byte-identical to the local domain/model.py. Any drift breaks
state_dict loading in the serving pipeline.
"""
import torch
import torch.nn as nn


# CIFAR-10 subset: indices 0, 1, 2, 3
CLASSES = ["airplane", "automobile", "bird", "cat"]
NUM_CLASSES = len(CLASSES)

CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)


class SmallCNN(nn.Module):
    """3-layer CNN with dropout. ~600K params."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

Writing /content/net.py


In [ ]:
import sys
sys.path.insert(0, '/content')
from net import SmallCNN

m = SmallCNN()
print("Parameters:", sum(p.numel() for p in m.parameters()))
# Expected: 618820

Parameters: 618820


In [ ]:
!grep "download=" /content/adversarial_training.py

    train_ds = datasets.CIFAR10(root="/content/data", train=True, download=True, transform=tfm)
    test_ds = datasets.CIFAR10(root="/content/data", train=False, download=True, transform=tfm)


In [6]:
%cd /content
!python adversarial_training.py

/content
Device: cpu
Training attacker: PGD-5 at eps up to 0.0314
Loss weights: clean=0.4, adv=0.6
Warmup: eps ramps from 0.0157 to 0.0314 over 5 epochs
Per-epoch eval on 200 samples; final eval on 500 samples

100% 170M/170M [33:56<00:00, 83.7kB/s]
Train batches: 141
Val batches: 16
epoch 01/20 | eps=0.0157 | loss=0.9374 (clean=0.8590 adv=0.9897) | val_acc=0.7385 | fgsm=0.5781 pgd=0.5508 | t=252s
  -> saved best (val_acc=0.7385)
epoch 02/20 | eps=0.0188 | loss=0.7615 (clean=0.6444 adv=0.8396) | val_acc=0.7670 | fgsm=0.5742 pgd=0.5742 | t=481s
  -> saved best (val_acc=0.7670)
epoch 03/20 | eps=0.0220 | loss=0.7148 (clean=0.5762 adv=0.8071) | val_acc=0.8110 | fgsm=0.6289 pgd=0.5859 | t=708s
  -> saved best (val_acc=0.8110)
epoch 04/20 | eps=0.0251 | loss=0.6787 (clean=0.5205 adv=0.7842) | val_acc=0.8275 | fgsm=0.6406 pgd=0.6172 | t=939s
  -> saved best (val_acc=0.8275)
epoch 05/20 | eps=0.0282 | loss=0.6545 (clean=0.4768 adv=0.7729) | val_acc=0.8305 | fgsm=0.6406 pgd=0.6055 | t=1168s
  

In [7]:
from google.colab import files
files.download('/content/baseline_robust.pt')
files.download('/content/robustness_curve.json')
files.download('/content/config_robust.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>